# 02 — Atari Pong: pixels → learned representation → policy

Now remove the convenient hand-written state vector.

Instead of:

```text
ball_x
ball_y
ball_velocity
paddle_y
```

the agent gets **pixels**.

A CNN learns useful visual features and the policy learns actions from those features.

This notebook is intended for **Google Colab** if you're on the Snapdragon Zenbook.

In [ ]:
!pip -q install "gymnasium[atari,accept-rom-license]" "stable-baselines3[extra]" "ale-py>=0.10" autorom
!AutoROM --accept-license >/dev/null 2>&1 || true


In [ ]:
import gymnasium as gym
import ale_py
import numpy as np
import matplotlib.pyplot as plt

gym.register_envs(ale_py)

# Try modern ALE id first.
env = gym.make("ALE/Pong-v5", render_mode="rgb_array")
obs, info = env.reset(seed=0)

print("Observation shape:", obs.shape)
print("Action count:", env.action_space.n)

plt.figure(figsize=(6,5))
plt.imshow(obs)
plt.axis("off")
plt.title("Atari Pong observation = pixels")
plt.show()


The important conceptual change:

```text
Toy Pong:
small vector -> MLP -> action

Atari Pong:
pixels -> CNN encoder -> latent features -> policy head -> action
```

The CNN's hidden activations are a **learned representation**. Nobody manually gives it `ball_x`.


In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack

venv = make_atari_env("PongNoFrameskip-v4", n_envs=4, seed=0)
venv = VecFrameStack(venv, n_stack=4)

model = PPO(
    "CnnPolicy",
    venv,
    verbose=1,
    n_steps=128,
    batch_size=256,
    tensorboard_log="./tb_atari/",
)

# Educational short run. Real competence takes much longer.
model.learn(total_timesteps=50_000)
model.save("pong_ppo_short")


## Why frame stacking?

One still image can show where the ball is, but not clearly which way it is moving.

Four recent frames make motion inferable.

That's already a tiny example of:

> **history can contain state information that one observation lacks.**

That connects directly to partial observability and eventually belief/state representations.


In [ ]:
# Quick evaluation rollout, storing frames for a short visualization.
eval_env = gym.make("ALE/Pong-v5", render_mode="rgb_array")
obs, _ = eval_env.reset(seed=1)

# This raw eval env doesn't match the VecFrameStack preprocessing used in training,
# so use this cell mainly to inspect the environment. For proper evaluation, use
# the same wrappers as training.
frames = [obs]

for _ in range(60):
    action = eval_env.action_space.sample()
    obs, reward, terminated, truncated, _ = eval_env.step(action)
    frames.append(obs)
    if terminated or truncated:
        obs, _ = eval_env.reset()

plt.figure(figsize=(6,5))
plt.imshow(frames[-1])
plt.axis("off")
plt.title("A later Atari frame")
plt.show()


### Don't overfocus on Atari training time

The reason to do this lab is to feel the jump:

```text
engineered state variables
        ↓
raw pixels
        ↓
learned representation
```

For the Sea of Stars lab, we initially cheat in a useful way: a VLM turns pixels into explicit
facts like `sword_active=True`. Later, you can replace those facts with learned visual embeddings.
